In [1]:
import json
import pandas as pd
import numpy as np
from collections import Counter

def squad2_to_df(path: str) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        squad = json.load(f)

    rows = []
    for article in squad["data"]:
        title = article["title"]
        for para_i, para in enumerate(article["paragraphs"]):
            context = para["context"]
            paragraph_id = f"{title}__{para_i}"

            for qa in para["qas"]:
                qid = qa["id"]
                question = qa["question"]
                is_impossible = qa.get("is_impossible", False)
                can_answer = "yes" if not is_impossible else "no"

                if can_answer == "yes" and len(qa.get("answers", [])) > 0:
                    answer_text  = qa["answers"][0]["text"]
                    answer_start = qa["answers"][0]["answer_start"]
                else:
                    answer_text  = "NO_ANSWER"
                    answer_start = -1

                rows.append({
                    "id":           qid,
                    "title":        title,
                    "paragraph_id": paragraph_id,
                    "context":      context,
                    "question":     question,
                    "can_answer":   can_answer,
                    "answer_text":  answer_text,
                    "answer_start": answer_start
                })

    return pd.DataFrame(rows)


def add_basic_features(df):
    df = df.copy()

    if "can_answer" in df.columns:
        df["y"] = (df["can_answer"].astype(str).str.lower() == "yes").astype(int)
    elif "is_impossible" in df.columns:
        df["y"] = (df["is_impossible"] == False).astype(int)
    else:
        df["y"] = (df["answer_text"] != "NO_ANSWER").astype(int)

    df["q_len_char"] = df["question"].astype(str).str.len()
    df["c_len_char"] = df["context"].astype(str).str.len()
    df["q_len_tok"]  = df["question"].astype(str).str.split().str.len()
    df["c_len_tok"]  = df["context"].astype(str).str.split().str.len()

    if "answer_text" in df.columns:
        df["a_len_char"] = df["answer_text"].astype(str).str.len()
        df["a_len_tok"]  = df["answer_text"].astype(str).str.split().str.len()
        df.loc[df["answer_text"] == "NO_ANSWER", ["a_len_char", "a_len_tok"]] = 0

    return df


train_df = squad2_to_df("train_sampled.json")
val_df   = squad2_to_df("val_sampled.json")
test_df  = squad2_to_df("test_sampled.json")

train_df = add_basic_features(train_df)
val_df   = add_basic_features(val_df)
test_df  = add_basic_features(test_df)

print("Train shape:", train_df.shape)
print("Val shape:  ", val_df.shape)
print("Test shape: ", test_df.shape)

Train shape: (8001, 15)
Val shape:   (996, 15)
Test shape:  (1004, 15)


In [2]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 384
DOC_STRIDE = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_df(df):
    questions   = df["question"].tolist()
    contexts    = df["context"].tolist()
    answers     = df["answer_text"].tolist()
    starts      = df["answer_start"].tolist()
    answerables = df["y"].tolist()

    tokenized = tokenizer(
        questions,
        contexts,
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map     = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions   = []
    end_positions     = []
    final_answerables = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answerable = answerables[sample_idx]
        final_answerables.append(answerable)

        if not answerable:
            start_positions.append(0)
            end_positions.append(0)
            continue

        answer_start = starts[sample_idx]
        answer_text  = answers[sample_idx]

        if answer_text == "NO_ANSWER" or answer_start == -1:
            start_positions.append(0)
            end_positions.append(0)
            continue

        answer_end   = answer_start + len(answer_text)
        sequence_ids = tokenized.sequence_ids(i)

        ctx_start = next(j for j, s in enumerate(sequence_ids) if s == 1)
        ctx_end   = len(sequence_ids) - 1
        while sequence_ids[ctx_end] != 1:
            ctx_end -= 1

        if offsets[ctx_start][0] > answer_start or offsets[ctx_end][1] < answer_end:
            start_positions.append(0)
            end_positions.append(0)
            continue

        start = ctx_start
        while start <= ctx_end and offsets[start][0] <= answer_start:
            start += 1
        start_positions.append(start - 1)

        end = ctx_end
        while end >= ctx_start and offsets[end][1] >= answer_end:
            end -= 1
        end_positions.append(end + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"]   = end_positions
    tokenized["answerable"]      = final_answerables
    return tokenized


print("Tokenizing train...")
train_tok = tokenize_df(train_df)
print("Tokenizing val...")
val_tok   = tokenize_df(val_df)
print("Tokenizing test...")
test_tok  = tokenize_df(test_df)

print(f"Train chunks: {len(train_tok['input_ids'])}")
print(f"Val chunks:   {len(val_tok['input_ids'])}")
print(f"Test chunks:  {len(test_tok['input_ids'])}")

/opt/anaconda3/envs/tf/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizing train...
Tokenizing val...
Tokenizing test...
Train chunks: 8080
Val chunks:   1001
Test chunks:  1004


In [3]:
train_dataset = Dataset.from_dict(train_tok)
val_dataset   = Dataset.from_dict(val_tok)
test_dataset  = Dataset.from_dict(test_tok)

cols = ["input_ids", "attention_mask", "token_type_ids", "start_positions", "end_positions"]
train_dataset.set_format(type="torch", columns=cols)
val_dataset.set_format(type="torch",   columns=cols)
test_dataset.set_format(type="torch",  columns=cols)

print("Train dataset:", len(train_dataset))
print("Val dataset:  ", len(val_dataset))
print("Test dataset: ", len(test_dataset))

Train dataset: 8080
Val dataset:   1001
Test dataset:  1004


In [4]:
import torch
from torch.utils.data import DataLoader
from transformers import DebertaV2ForQuestionAnswering, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score

EPOCHS     = 3
BATCH_SIZE = 8
LR         = 2e-5
SAVE_PATH  = "deberta_squad_finetuned"

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu" # we are using GPU if available, otherwise CPU
)
print(f"Using device: {device}")

# Carica modello in float32
model = DebertaV2ForQuestionAnswering.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
model.to(device)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)

optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.2 * total_steps),
    num_training_steps=total_steps
)

best_val_loss = float("inf")
patience      = 0
MAX_PATIENCE  = 2

for epoch in range(EPOCHS):
    model.train()
    total_loss  = 0
    step_preds  = []
    step_labels = []

    for step, batch in enumerate(train_loader):
        batch   = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss    = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        with torch.no_grad():
            pred_starts = outputs.start_logits.argmax(dim=-1)
            preds  = (pred_starts != 0).long().cpu().tolist()
            labels = (batch["start_positions"] != 0).long().cpu().tolist()
            step_preds.extend(preds)
            step_labels.extend(labels)

        total_loss += loss.item()

        if step % 50 == 0 and step > 0:
            f1 = f1_score(step_labels, step_preds, average="macro", zero_division=0)
            print(f"Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {loss.item():.4f} | F1: {f1:.4f}")
            step_preds  = []
            step_labels = []

    avg_train_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss   = 0
    val_preds  = []
    val_labels = []

    with torch.no_grad():
        for batch in val_loader:
            batch   = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            val_loss += outputs.loss.item()

            pred_starts = outputs.start_logits.argmax(dim=-1)
            preds  = (pred_starts != 0).long().cpu().tolist()
            labels = (batch["start_positions"] != 0).long().cpu().tolist()
            val_preds.extend(preds)
            val_labels.extend(labels)

    avg_val_loss = val_loss / len(val_loader)
    val_f1       = f1_score(val_labels, val_preds, average="macro", zero_division=0)

    print(f"\n=== Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val F1: {val_f1:.4f} ===\n")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience      = 0
        model.save_pretrained(SAVE_PATH)
        tokenizer.save_pretrained(SAVE_PATH)
        print(f"=> Modello salvato (val loss: {best_val_loss:.4f})\n")
    else:
        patience += 1
        print(f"=> No improvement. Patience: {patience}/{MAX_PATIENCE}\n")
        if patience >= MAX_PATIENCE:
            print("Early stopping.")
            break

Using device: mps


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 577.40it/s]
DebertaV2ForQuestionAnswering LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
qa_outputs.weight                       | MISSING    | 
qa_outputs.bias                         | MISSING    | 

Notes:
- UNEXPECT

Epoch 1 | Step 50/1010 | Loss: 5.5592 | F1: 0.3498
Epoch 1 | Step 100/1010 | Loss: 4.5139 | F1: 0.4690
Epoch 1 | Step 150/1010 | Loss: 3.3117 | F1: 0.4874
Epoch 1 | Step 200/1010 | Loss: 2.6379 | F1: 0.3747
Epoch 1 | Step 250/1010 | Loss: 2.8137 | F1: 0.3333
Epoch 1 | Step 300/1010 | Loss: 2.0738 | F1: 0.3869
Epoch 1 | Step 350/1010 | Loss: 1.5605 | F1: 0.3717
Epoch 1 | Step 400/1010 | Loss: 2.9451 | F1: 0.3773
Epoch 1 | Step 450/1010 | Loss: 2.7868 | F1: 0.4107
Epoch 1 | Step 500/1010 | Loss: 1.0835 | F1: 0.4557
Epoch 1 | Step 550/1010 | Loss: 2.6390 | F1: 0.4881
Epoch 1 | Step 600/1010 | Loss: 2.6206 | F1: 0.4708
Epoch 1 | Step 650/1010 | Loss: 2.0126 | F1: 0.5312
Epoch 1 | Step 700/1010 | Loss: 1.2572 | F1: 0.5200
Epoch 1 | Step 750/1010 | Loss: 2.2475 | F1: 0.5582
Epoch 1 | Step 800/1010 | Loss: 1.6920 | F1: 0.5156
Epoch 1 | Step 850/1010 | Loss: 1.3679 | F1: 0.5681
Epoch 1 | Step 900/1010 | Loss: 2.0202 | F1: 0.6134
Epoch 1 | Step 950/1010 | Loss: 1.4750 | F1: 0.6777
Epoch 1 | Ste

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.14s/it]


=> Modello salvato (val loss: 1.3474)

Epoch 2 | Step 50/1010 | Loss: 1.7712 | F1: 0.7461
Epoch 2 | Step 100/1010 | Loss: 1.8348 | F1: 0.7269
Epoch 2 | Step 150/1010 | Loss: 1.2119 | F1: 0.7571
Epoch 2 | Step 200/1010 | Loss: 1.5965 | F1: 0.7430
Epoch 2 | Step 250/1010 | Loss: 1.5946 | F1: 0.7947
Epoch 2 | Step 300/1010 | Loss: 1.3247 | F1: 0.7950
Epoch 2 | Step 350/1010 | Loss: 0.5711 | F1: 0.7825
Epoch 2 | Step 400/1010 | Loss: 1.6418 | F1: 0.7498
Epoch 2 | Step 450/1010 | Loss: 0.6170 | F1: 0.7624
Epoch 2 | Step 500/1010 | Loss: 2.0894 | F1: 0.7525
Epoch 2 | Step 550/1010 | Loss: 1.0798 | F1: 0.8025
Epoch 2 | Step 600/1010 | Loss: 0.8436 | F1: 0.8074
Epoch 2 | Step 650/1010 | Loss: 1.0069 | F1: 0.8050
Epoch 2 | Step 700/1010 | Loss: 0.8640 | F1: 0.8149
Epoch 2 | Step 750/1010 | Loss: 0.7878 | F1: 0.8300
Epoch 2 | Step 800/1010 | Loss: 1.7128 | F1: 0.8270
Epoch 2 | Step 850/1010 | Loss: 1.0060 | F1: 0.8598
Epoch 2 | Step 900/1010 | Loss: 0.4818 | F1: 0.8316
Epoch 2 | Step 950/1010 | 

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.06it/s]


=> Modello salvato (val loss: 1.0869)

Epoch 3 | Step 50/1010 | Loss: 0.4973 | F1: 0.9117
Epoch 3 | Step 100/1010 | Loss: 0.4447 | F1: 0.9050
Epoch 3 | Step 150/1010 | Loss: 1.0151 | F1: 0.8698
Epoch 3 | Step 200/1010 | Loss: 0.6840 | F1: 0.8900
Epoch 3 | Step 250/1010 | Loss: 0.4711 | F1: 0.9072
Epoch 3 | Step 300/1010 | Loss: 1.6186 | F1: 0.9069
Epoch 3 | Step 350/1010 | Loss: 0.9631 | F1: 0.8872
Epoch 3 | Step 400/1010 | Loss: 0.8030 | F1: 0.9048
Epoch 3 | Step 450/1010 | Loss: 0.1971 | F1: 0.8920
Epoch 3 | Step 500/1010 | Loss: 0.4636 | F1: 0.9225
Epoch 3 | Step 550/1010 | Loss: 0.4250 | F1: 0.9225
Epoch 3 | Step 600/1010 | Loss: 0.2238 | F1: 0.9350
Epoch 3 | Step 650/1010 | Loss: 0.6981 | F1: 0.8947
Epoch 3 | Step 700/1010 | Loss: 0.6580 | F1: 0.9124
Epoch 3 | Step 750/1010 | Loss: 0.4445 | F1: 0.9074
Epoch 3 | Step 800/1010 | Loss: 0.5938 | F1: 0.9521
Epoch 3 | Step 850/1010 | Loss: 0.0648 | F1: 0.9298
Epoch 3 | Step 900/1010 | Loss: 0.2729 | F1: 0.9093
Epoch 3 | Step 950/1010 | 